# RAG Fundamentals: retrieve, then generate — built from scratch

Ask a frozen language model *"When was the Helios-7 satellite launched?"* and, if Helios-7 is a private project it never saw, it either refuses or **confidently invents a date**. The model's factual world froze the day pretraining ended, and no prompting puts a fact into weights that never saw it.

**Retrieval-Augmented Generation (RAG)** is the fix: don't bake the knowledge into the model — **fetch the relevant passages at question time and put them in the prompt.** It's a closed-book exam turned open-book. This notebook builds the whole pipeline from primitives — a transparent embedder, cosine top-k retrieval, prompt augmentation — so you can *see* retrieval pick the right passage by eye.

**By the end you'll have:** embedded a tiny private corpus into an index, retrieved the top-k passages for a question the model can't answer from memory, *seen* the grounded vs ungrounded answers diverge, and measured the recall/noise tradeoff as k grows. We import the canonical functions from [`rag_fundamentals.py`](rag_fundamentals.py) so the notebook and the script share the exact same code — and because the embedder is **hash-deterministic** (a fixed FNV-1a hash, no random step), every number below is identical on every run and machine.

## Step 1 — set up: import the canonical RAG functions

Everything below uses the verified functions from `rag_fundamentals.py` — the same code the concept page shows — so there's no drift between page, script, and notebook. We print the numpy version and (if torch is present) the device, then show the eight-passage corpus. **Read the corpus:** four Helios-7 facts the model was never trained on (the *private* knowledge), plus four generic facts. Retrieval will have to pick the right one for each question.

In [1]:
import numpy as np

from rag_fundamentals import (
    CORPUS, PRIVATE_QUESTION, TOP_K,
    compute_idf, build_index, embed, cosine_top_k,
    build_prompt, generate_ungrounded, rag_answer,
)

print('numpy:', np.__version__)
try:
    import torch
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    print('torch:', torch.__version__, '| device:', device, '(retrieval itself is pure numpy)')
except ImportError:
    print('torch: not installed (retrieval is pure numpy — unaffected)')

print(f'\ncorpus: {len(CORPUS)} passages | top_k = {TOP_K}')
for i, passage in enumerate(CORPUS):
    print(f'  doc[{i}] {passage}')

numpy: 2.4.6


torch: 2.12.0 | device: mps (retrieval itself is pure numpy)

corpus: 8 passages | top_k = 3
  doc[0] The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
  doc[1] Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
  doc[2] The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.
  doc[3] Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
  doc[4] Photosynthesis converts carbon dioxide and water into glucose using sunlight.
  doc[5] The Eiffel Tower in Paris was completed in 1889 for the World's Fair.
  doc[6] A standard chessboard has 64 squares arranged in an 8 by 8 grid.
  doc[7] Water boils at 100 degrees Celsius at standard atmospheric pressure.


## Step 2 — how text becomes a vector (IDF-weighted hashing)

Retrieval matches on *vectors*, so first we need text → vector. Our embedder is deliberately **transparent** (no model download): it tokenizes, weights each token by **IDF** (rare words like "Helios" matter more than "the"), hashes each token to a slot, and **L2-normalizes** the result. Normalizing is the key trick — it makes cosine similarity collapse to a plain dot product. Let's embed one passage and confirm its shape and unit norm.

In [2]:
idf = compute_idf(CORPUS)                 # log(N/df) per token — rare words weigh more
v = embed(CORPUS[0], idf)                 # embed the first passage
print('embedding shape:', v.shape)        # (256,)
print('L2 norm (should be ~1.0):', round(float(np.linalg.norm(v)), 6))

# The vector is SPARSE — only the slots a token hashed to are non-zero. Show those real weights
# (slot index -> value) so you can see "text became a vector", not a row of empty leading slots.
nonzero_slots = np.nonzero(v)[0]
print(f'non-zero components: {len(nonzero_slots)} of {v.shape[0]} slots filled')
for slot in nonzero_slots[:6]:            # first few filled slots, each = one hashed token's IDF weight
    print(f'  slot {slot:>3}: {v[slot]:+.3f}')

embedding shape: (256,)
L2 norm (should be ~1.0): 1.0
non-zero components: 12 of 256 slots filled
  slot  23: -0.145
  slot  28: +0.602
  slot  76: +0.263
  slot 102: +0.263
  slot 117: -0.263
  slot 185: -0.263


## Step 3 — build the index (embed every passage once)

The **index** is just every passage embedded into one `(n_docs, dim)` matrix — built **once, offline**. Each row is a unit vector, so later a single matrix-vector product scores the whole corpus against a query. We assert the shape and that every row is unit-norm *before* trusting any retrieval result — a cheap check that catches the most common embedding bugs.

In [3]:
index = build_index(CORPUS, idf)
print('index shape (n_docs, dim):', index.shape)
norms = np.linalg.norm(index, axis=1)
assert index.shape == (len(CORPUS), v.shape[0]), 'index must be (n_docs, embed_dim)'
assert np.allclose(norms, 1.0, atol=1e-9), 'every row must be unit-norm'
print('all rows unit-norm:', bool(np.allclose(norms, 1.0, atol=1e-9)))

index shape (n_docs, dim): (8, 256)
all rows unit-norm: True


## Step 4 — retrieve: see it by eye

Now the heart of RAG. Embed the question with the **same** embedder, then `cosine_top_k` scores every passage with one dot-product and returns the top-3. **Read the scores below:** the answering passage `doc[0]` should win by a wide margin because it shares the rare tokens "helios", "satellite", "launched" with the query. Watch which passages get pulled in — and notice the third one is a near-miss distractor (the kind a re-ranker later removes).

In [4]:
print('QUESTION:', PRIVATE_QUESTION, '\n')
q_vec = embed(PRIVATE_QUESTION, idf)
top_idx, top_scores = cosine_top_k(q_vec, index, k=TOP_K)
for rank, (i, s) in enumerate(zip(top_idx, top_scores), 1):
    print(f'{rank}. doc[{i}]  cos={s:.3f}  | {CORPUS[i]}')

# correctness BEFORE any claim: the answering passage must be retrieved, at rank 1
assert top_idx[0] == 0, 'the gold passage doc[0] must rank #1 for this query'
print('\ngold passage doc[0] retrieved at rank 1:', bool(top_idx[0] == 0))

QUESTION: When was the Helios-7 satellite launched? 

1. doc[0]  cos=0.638  | The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
2. doc[2]  cos=0.282  | The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.
3. doc[5]  cos=0.213  | The Eiffel Tower in Paris was completed in 1889 for the World's Fair.

gold passage doc[0] retrieved at rank 1: True


## Step 5 — the headline contrast: grounded vs ungrounded

This is the whole point of RAG in two lines. **Without retrieval**, the model answers from parametric memory — and it has *no* Helios-7 facts, so the honest outcome is "I don't know" (a real LLM would often hallucinate a fake date, which is worse). **With retrieval**, the right passage sits in the prompt, so the answer is correct *and* traceable to `doc[0]`. The assert proves the grounded answer contains the real date and the ungrounded one cannot.

In [5]:
ungrounded = generate_ungrounded(PRIVATE_QUESTION)
grounded, _, _ = rag_answer(PRIVATE_QUESTION, CORPUS, idf, index)
print('UNGROUNDED (memory only):', ungrounded.text)
print('GROUNDED  (retrieve+gen):', grounded.text)

assert 'March 3rd, 2024' in grounded.text, 'grounded answer must contain the real launch date'
assert 'March 3rd, 2024' not in ungrounded.text, 'ungrounded path cannot know the private date'
print('\ngrounded has the correct date; ungrounded does not:',
      bool('March 3rd, 2024' in grounded.text and 'March 3rd, 2024' not in ungrounded.text))

UNGROUNDED (memory only): I don't have information about that.
GROUNDED  (retrieve+gen): The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.

grounded has the correct date; ungrounded does not: True


## Step 6 — the augmented prompt (the "open book on the desk")

What does the generator actually *see*? `build_prompt` stitches the retrieved passages and the question into one prompt with an instruction to answer **only** from the context. This block — not just the bare question — is what the LLM generates from. That stitching *is* the "augmented" in Retrieval-Augmented Generation.

In [6]:
retrieved = [CORPUS[i] for i in top_idx]
print(build_prompt(PRIVATE_QUESTION, retrieved))

Answer the question using ONLY the context below. If the context does not contain the answer, say you don't know.

Context:
[1] The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
[2] The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.
[3] The Eiffel Tower in Paris was completed in 1889 for the World's Fair.

Question: When was the Helios-7 satellite launched?
Answer:


## Step 7 — measure the recall/noise knob (recall@k)

`k` is the central retrieval knob. To feel the tradeoff, we probe with **paraphrased** questions that barely share words with their answering passage (e.g. "liftoff date" vs the passage's "launched on") — the realistic case where a lexical embedder doesn't always rank the right passage first. **Predict before you run:** at k=1 will recall be perfect, or will the paraphrases trip it up? Then read the curve climb as k grows.

In [7]:
probes = [
    ('What is the liftoff date of the orbiter from Kourou?', 0),
    ('Which optical sensor does the spacecraft carry?', 1),
    ('Who is the engineer in charge of the programme?', 2),
    ('How long does a single revolution around the planet take?', 3),
    ('Total number of playing tiles on a chess grid?', 6),
    ('Boiling point of liquid at sea level pressure?', 7),
]
print(f"{'k':>3} | recall@k")
print('-' * 16)
for k in range(1, len(CORPUS) + 1):
    hits = sum(gold in cosine_top_k(embed(q, idf), index, k=k)[0] for q, gold in probes)
    print(f'{k:>3} | {hits / len(probes):.2f}')

  k | recall@k
----------------
  1 | 0.50
  2 | 0.67
  3 | 0.67
  4 | 0.67
  5 | 0.83
  6 | 0.83
  7 | 0.83
  8 | 1.00


## Try it yourself

Before you change anything, **predict**: if you make the embedding **less lexical** — say by adding a synonym to the corpus, or lowering `EMBED_DIM` so more tokens *collide* into the same slot — does recall@1 go **up** or **down**?

Then try it: re-import with a smaller dim, or add a probe whose wording shares *zero* content words with its passage, and watch recall@1 drop. *Hint:* a purely lexical embedder can only match words it literally shares — paraphrases and synonyms are exactly where it fails, which is **why learned semantic embeddings exist** (the next chapter). The fix for a low recall@1 is almost never a smarter LLM; it's a better **retriever**.

> To see the real thing, swap our hashing embedder for a `sentence-transformers` model and re-run Step 7 — the paraphrases that miss here will start matching, lifting the left end of the curve.

## What we saw

- **Text became vectors, and retrieval became geometry** — `index @ query` scored all eight passages at once, and the answering `doc[0]` won by a wide cosine margin (it shares the rare, high-IDF tokens with the query).
- **The grounded answer was correct and traceable; the ungrounded one couldn't know** — RAG didn't make the model *smarter*, it gave it the *right page*.
- **The augmented prompt is the whole mechanism** — instruction + retrieved context + question is what the LLM generates from.
- **k trades recall for noise** — recall@1 was imperfect on paraphrased queries and climbed with k; better embeddings and re-ranking lift the left end of that curve.

**What we skipped** (and where to read it): better **chunking** so facts aren't split ([concept 2](../../02-Document-Chunking-Strategies/02-Document-Chunking-Strategies.md)); learned **semantic embeddings** so paraphrases match ([concept 3](../../03-Embedding-Models-for-Retrieval/03-Embedding-Models-for-Retrieval.md)); production **vector indexes**, **hybrid search**, and **re-ranking**; and the **lost-in-the-middle** and **stale-index** failure modes. All of that is in the [RAG Fundamentals concept page](../01-RAG-Fundamentals.md), with its [curated references](../01-RAG-Fundamentals.references.md).